# Imports

In [ ]:
# # 1. Install updated libraries
# !pip install -q git+https://github.com/WildlifeDatasets/wildlife-datasets@develop
# !pip install -q git+https://github.com/WildlifeDatasets/wildlife-tools
# !pip install -q transformers==4.40.2 accelerate==0.30.1

In [ ]:
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from wildlife_datasets.datasets import AnimalCLEF2026

from src.feature_extraction.wildfusion import WildFusionFeatureExtractor
from src.clustering_algorithms.chinese_whispers import chinese_whispers
from src.clustering_algorithms.leiden import leiden_clustering
from src.utilities.image2identity import *
from src.utilities.reranking import re_ranking

# Setups

In [ ]:
# Setup Paths and Metadata
root = 'Dataset_Root'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#Loading the 2026 dataset
dataset = AnimalCLEF2026(root, load_label=True)

# Splits
dataset_database = dataset.get_subset(dataset.metadata['split'] == 'train')
dataset_query = dataset.get_subset(dataset.metadata['split'] == 'test')

print(f"Database (Train) size: {len(dataset_database)}")
print(f"Test (Query) size: {len(dataset_query)}")

In [ ]:
# Use a subset for calibration (Generic/No species-specific prep during global fit)
dataset_calibration = dataset_database.get_subset(range(min(512, len(dataset_database)))) 

wildfusion = WildFusionFeatureExtractor(dataset_calibration, device=device)
wildfusion.calibrate()

In [ ]:
all_results = {}

ds_name = "SalamanderID2025"
database_dir = "DataBase"

save_dir = Path(database_dir)
save_dir.mkdir(parents=True, exist_ok=True)

print(f"\n--- Processing Dataset: {ds_name} ---")

ds_train = dataset_database.get_subset(
    dataset_database.metadata['dataset'] == ds_name
)

if len(ds_train) == 0:
    print("Skipping (no train images)")

print(f"Using FULL dataset size: {len(ds_train)}")

ds_train_subset = ds_train
indices = np.arange(len(ds_train))  

sim_eval = wildfusion.run(ds_name, data=(ds_train_subset, ds_train_subset), B=256)

sim = sim_eval.copy()
sim[np.isinf(sim)] = 0
sim = np.clip(sim, 0, 1)

all_results[ds_name] = {
    "sim": sim,
    "indices": indices
}

np.save(f"{database_dir}/sim_matrix_{ds_name}_full.npy", all_results)